# Обучение модели logP

Данные из моего проекта [logP-Prediction](https://github.com/SuperStas22/logP-Prediction) -по сути расторимость:

* `logP-14000.csv`: ~14 тыс. молекул с экспериментальным logP (log K<sub>ow</sub>) и RDKit-дескрипторами, посчитанными по SMILES;
* `logP-707-Martel.csv`: внешний тест Martel (707 молекул), с обучающей выборкой не пересекается.

Модель: градиентный бустинг - были и лучше модели, но эта для прсототы

Порог в метаданных: правило пяти Липински, logP ≤ 5. Сервис отдаёт число и флаг `lipinski_ok` - Правило старое и эмпирическое, можно ли считать молекулу лекарственно подобной.

In [1]:
import json
import platform
from datetime import date
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

SEED = 42
DATA = Path("../data")  
ARTIFACT = Path("../artifact/model.joblib")

MODEL_VERSION = "hgb-1.0"
THRESHOLD = 5.0  # правило Липински

FEATURES = [
    "MolWt",
    "HeavyAtomCount",
    "NumHeteroatoms",
    "fr_halogen",
    "NumHDonors",
    "NumHAcceptors",
    "TPSA",
    "MolMR",
    "NumRotatableBonds",
    "RingCount",
    "NumAromaticRings",
    "FractionCSP3",
    "MaxPartialCharge",
    "MinPartialCharge",
] ## в этом плане не очень очеивино :)

In [2]:
def load(name: str) -> pd.DataFrame:
    local = DATA / name
    return pd.read_csv(local)


df = load("logP-14000.csv").rename(columns={"Kow": "logP"})
martel = load("logP-707-Martel.csv").rename(columns={"logPow": "logP"})

print("строк:", len(df), "дубликатов SMILES:", df.SMILES.duplicated().sum(), "пересечение с Martel:", df.SMILES.isin(martel.SMILES).sum())
df = df.drop_duplicates("SMILES").reset_index(drop=True)
df[FEATURES + ["logP"]].describe().T[["count", "mean", "min", "max"]].round(3)

строк: 14176 дубликатов SMILES: 24 пересечение с Martel: 0


,count,mean,min,max
MolWt,14152.0,253.463,40.065,1202.635
HeavyAtomCount,14152.0,17.173,3.000,85.000
NumHeteroatoms,14152.0,5.030,0.000,24.000
fr_halogen,14152.0,0.575,0.000,23.000
NumHDonors,14152.0,1.118,0.000,10.000
NumHAcceptors,14152.0,3.493,0.000,20.000
TPSA,14152.0,58.910,0.000,288.280
MolMR,14152.0,66.305,5.007,341.580
NumRotatableBonds,14152.0,3.410,0.000,32.000
RingCount,14152.0,1.760,0.000,9.000


In [3]:
na = df[FEATURES].isna().sum()
print(na[na > 0])
df.loc[df.MaxPartialCharge.isna(), "SMILES"].head(5).tolist()

MaxPartialCharge    13
MinPartialCharge    13
dtype: int64


['O=[As](O)(O)c1ccccc1',
 'c1ccc2[se]ccc2c1',
 'c1ccc2n[se]nc2c1',
 'O=[IH2]c1ccccc1',
 'C[As](=O)(O)O']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(df[FEATURES], df["logP"], test_size=0.2, random_state=SEED)

pipeline = Pipeline(
    [
        ("prep", ColumnTransformer([("num", SimpleImputer(strategy="median"), FEATURES)])),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=300, learning_rate=0.1, max_leaf_nodes=31, l2_regularization=1.0, early_stopping=False, random_state=SEED
            ),
        ),
    ]
)
pipeline.fit(x_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['MolWt', 'HeavyAtomCount',
                                                   'NumHeteroatoms',
                                                   'fr_halogen', 'NumHDonors',
                                                   'NumHAcceptors', 'TPSA',
                                                   'MolMR', 'NumRotatableBonds',
                                                   'RingCount',
                                                   'NumAromaticRings',
                                                   'FractionCSP3',
                                                   'MaxPartialCharge',
                                                   'MinPartialCharge'])])),
                ('model',
                 HistGradientBoostingRegressor(early_stopping=False,
                                               l2_regularization=1.0,
                                               max_iter=300,
                                               random_state=42))])

In [5]:
def scores(y, p) -> dict:
    return {
        "R2": round(r2_score(y, p), 3),
        "MAE": round(mean_absolute_error(y, p), 3),
        "RMSE": round(root_mean_squared_error(y, p), 3),
        "n": len(y),
    }


report = pd.DataFrame(
    {
        "holdout, бустинг": scores(y_test, pipeline.predict(x_test)),
        "holdout, Crippen MolLogP": scores(y_test, df.loc[x_test.index, "MolLogP"]),
        "Martel, бустинг": scores(martel.logP, pipeline.predict(martel[FEATURES])),
        "Martel, Crippen MolLogP": scores(martel.logP, martel.MolLogP),
    }
).T
report

,R2,MAE,RMSE,n
"holdout, бустинг",0.879,0.452,0.637,2831.0
"holdout, Crippen MolLogP",0.775,0.608,0.868,2831.0
"Martel, бустинг",-0.530,1.193,1.458,707.0
"Martel, Crippen MolLogP",-0.085,0.955,1.227,707.0


In [6]:
pred = pipeline.predict(x_test)
flag_true, flag_pred = y_test <= THRESHOLD, pred <= THRESHOLD
print(f"совпадение флага на holdout: {(flag_true == flag_pred).mean():.3f}, доля logP > 5 в данных: {(df.logP > THRESHOLD).mean():.3f}")

совпадение флага на holdout: 0.980, доля logP > 5 в данных: 0.057


## Артефакт: pipeline + metadata

In [7]:
metadata = {
    "model_version": MODEL_VERSION,
    "features": FEATURES,
    "threshold": THRESHOLD,
    "target": "logP",
    "trained_at": date.today().isoformat(),
    "n_train": len(x_train),
    "metrics": report.drop(columns="n").to_dict("index"),
    "sklearn": sklearn.__version__,
    "python": platform.python_version(),
}
ARTIFACT.parent.mkdir(exist_ok=True)
joblib.dump({"pipeline": pipeline, "metadata": metadata}, ARTIFACT, compress=3)
print(ARTIFACT, f"{ARTIFACT.stat().st_size / 1024:.0f} KB")

../artifact/model.joblib 457 KB


пример запроса ибупрофен, эксперимент 3.97, даёт разумное число. Этот же пример лежит в `good.json`.

In [8]:
bundle = joblib.load(ARTIFACT)
ibuprofen = df[df.SMILES == "CC(C)Cc1ccc(C(C)C(=O)O)cc1"].iloc[0]
row = {f: ibuprofen[f].item() for f in FEATURES}
print("эксперимент:", ibuprofen.logP, "предсказание:", round(float(bundle["pipeline"].predict(pd.DataFrame([row]))[0]), 3))

row = {f: (round(v, 4) if isinstance(v, float) else v) for f, v in row.items()}
Path("../good.json").write_text(json.dumps(row, indent=2) + "\n")
row

эксперимент: 3.97 предсказание: 4.06


{'MolWt': 206.285,
 'HeavyAtomCount': 15,
 'NumHeteroatoms': 2,
 'fr_halogen': 0,
 'NumHDonors': 1,
 'NumHAcceptors': 1,
 'TPSA': 37.3,
 'MolMR': 61.0348,
 'NumRotatableBonds': 4,
 'RingCount': 1,
 'NumAromaticRings': 1,
 'FractionCSP3': 0.4615,
 'MaxPartialCharge': 0.3102,
 'MinPartialCharge': -0.4808}